In [ ]:
import torch

In [ ]:
torch.cuda.device(0)

In [ ]:
torch.cuda.get_device_name()

In [ ]:
torch.cuda.is_available()

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    print("GPU not available, using CPU instead.")

In [ ]:
num_epochs = 5
batch_size = 100
learning_rate = 0.001

In [ ]:
import torchvision.datasets as dsets
import torchvision.transforms as transforms

In [ ]:
train_dataset = dsets.MNIST(
    root="./data", train=True, transform=transforms.ToTensor(), download=True
)
test_dataset = dsets.MNIST(
    root="./data", train=False, transform=transforms.ToTensor(), download=True
)

In [ ]:
import torch.utils.data as Data

In [ ]:
train_loader = Data.DataLoader(
    dataset=train_dataset, batch_size=batch_size, shuffle=True
)

test_loader = Data.DataLoader(
    dataset=test_dataset, batch_size=batch_size, shuffle=False
)

In [ ]:
import torch.nn as nn

In [ ]:
class Model(nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.dense = nn.Sequential(
            nn.Linear(14 * 14 * 128, 1024), nn.ReLU(), nn.Linear(1024, 10)
        )

    def forward(self, x):
        out = self.conv(x)
        out = out.view(-1, 14 * 14 * 128)
        out = self.dense(out)
        return out

In [ ]:
model = Model()
model.cuda()

In [ ]:
import torch.optim as opt

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = opt.Adam(model.parameters(), lr=learning_rate)

In [ ]:
from torch.autograd import Variable

In [ ]:
for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_loader):
        images = Variable(images.type(torch.cuda.FloatTensor))
        labels = Variable(labels.type(torch.cuda.LongTensor))

        optimizer.zero_grad()
        outputs = model(images)

        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        if (i + 1) % 100 == 0:
            print("Epoch: %d, Batch: %d, Loss: %.4f" % (epoch + 1, i + 1, loss.data))

In [ ]:
import torch

In [ ]:
correct = 0
total = 0
for images, labels in test_loader:
    images = Variable(images.type(torch.cuda.FloatTensor))
    outputs = model(images)
    _, pred = torch.max(outputs.data, 1)
    correct += (pred == labels.type(torch.cuda.LongTensor)).sum()
    total += labels.size(0)
print("Accuracy: %.3f %%" % (100.0 * float(correct) / float(total)))